# Machine Learning Foundations
## Clinical Data Structures & Binary Classification for Heart Disease

---

### Why Clinical ML Is Different

Clinical datasets are unlike the clean, synthetic datasets in most ML tutorials. They contain **missing values** (patients who skipped tests), **mixed data types** (numeric vitals alongside categorical diagnoses), **class imbalance** (disease is often rare), and **domain-specific encoding** that only makes sense if you understand the medicine.

Getting clinical ML wrong has real consequences — a model that misclassifies a high-risk patient as healthy is not just a low-accuracy model; it is a patient who doesn't receive treatment.

This notebook uses the **UCI Heart Disease dataset**, a landmark dataset collected from four hospitals across the US, Hungary, and Switzerland. It has been used in hundreds of published studies since 1988. By the end, you will:

1. Understand the structure and quirks of real clinical data
2. Know how to encode, clean, and engineer features from clinical records
3. Build binary classifiers that predict the **presence or absence of heart disease**
4. Evaluate those classifiers using clinically appropriate metrics

---

### Dataset Overview

| Field | Description | Type |
|---|---|---|
| `age` | Patient age in years | Continuous |
| `sex` | Biological sex (Male/Female) | Binary categorical |
| `dataset` | Originating hospital/study | Nominal categorical |
| `cp` | Chest pain type (4 categories) | Ordinal categorical |
| `trestbps` | Resting blood pressure (mmHg) | Continuous |
| `chol` | Serum cholesterol (mg/dL) | Continuous |
| `fbs` | Fasting blood sugar > 120 mg/dL | Binary categorical |
| `restecg` | Resting ECG results (3 categories) | Nominal categorical |
| `thalch` | Maximum heart rate achieved | Continuous |
| `exang` | Exercise-induced angina | Binary categorical |
| `oldpeak` | ST depression induced by exercise | Continuous |
| `slope` | Slope of peak exercise ST segment | Ordinal categorical |
| `ca` | Number of major vessels coloured by fluoroscopy | Discrete numeric |
| `thal` | Thalassemia type | Nominal categorical |
| `num` | **Target** — diagnosis (0 = no disease; 1–4 = disease severity) | Ordinal |

**Our ML task:** Predict whether a patient has heart disease (`num > 0`) — a binary classification problem.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings

from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier, export_text
from sklearn.ensemble        import RandomForestClassifier
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.pipeline        import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics         import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
    ConfusionMatrixDisplay, classification_report
)
from sklearn.impute          import SimpleImputer

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

# ── Load raw data ──
df_raw = pd.read_csv('heart_disease_uci.csv')
print(f'Shape: {df_raw.shape[0]} patients × {df_raw.shape[1]} columns')
print(f'Hospitals: {df_raw["dataset"].value_counts().to_dict()}')
print(f'Target (num) values: {df_raw["num"].value_counts().sort_index().to_dict()}')
df_raw.head()

---
## Part 1 — Introduction to Clinical Data Structures

### 1.1 What Makes a Clinical Dataset

A clinical record is a structured snapshot of a patient's physiological state at a specific moment in time. Each **row** is a patient encounter; each **column** is a clinical variable — called a **feature** in ML or a **covariate** in statistics.

Clinical features fall into four categories that require different handling:

**1. Continuous measurements** — numeric values on a scale with meaningful distance between points. Blood pressure (120 mmHg vs 140 mmHg is a real 20-unit gap). These are ready for most ML algorithms but often need scaling.

**2. Binary flags** — yes/no clinical observations. Fasting blood sugar above threshold, exercise-induced angina, diabetes presence. Already numeric (0/1) or easily encoded.

**3. Nominal categories** — unordered labels. Hospital site (Cleveland, Hungary), chest pain type (typical angina vs atypical angina), thalassemia type. Must be one-hot encoded — you cannot assign them arbitrary numbers because that would imply a false ordering.

**4. Ordinal categories** — ordered labels where rank matters but the gaps may not be equal. Slope of the ST segment (upsloping < flat < downsloping in clinical risk). These can be label-encoded (0, 1, 2) or one-hot encoded depending on context.

---

### 1.2 Missing Data in Clinical Records

Missing clinical data is not random — it carries information. A missing fluoroscopy result (`ca`) often means the test was not ordered, which itself could be a clinical signal. There are three standard mechanisms:

- **MCAR (Missing Completely At Random):** The probability of missing is unrelated to any variable. Safe to impute or drop.
- **MAR (Missing At Random):** The probability of missing depends on *other observed* variables, not the missing value itself. Imputation with observed features is valid.
- **MNAR (Missing Not At Random):** The probability of missing depends on the *unobserved value itself* — the most dangerous. A missing thalassemia result might mean the test was refused by a very high-risk patient.

In this dataset, `ca` and `thal` are missing in over 50% of rows — almost exclusively in non-Cleveland sites. This is **site-level MNAR**: those hospitals did not routinely perform these tests.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1.3  Audit the raw clinical data
# Understand every column: type, range, missingness
# ─────────────────────────────────────────────────────────────

df = df_raw.copy()

# Fix boolean-like object columns that were read as strings
for col in ['fbs', 'exang']:
    df[col] = df[col].map({True: 1, False: 0, 'TRUE': 1, 'FALSE': 0,
                           'True': 1, 'False': 0}).astype('Int64')

print('=== Clinical Feature Audit ===')
print(f'{"Column":12s}  {"Type":10s}  {"Missing":>8}  {"Missing%":>9}  {"Unique":>7}  Sample Values')
print('─' * 80)
for col in df.columns:
    n_miss  = df[col].isnull().sum()
    pct     = n_miss / len(df) * 100
    dtype   = str(df[col].dtype)
    n_uniq  = df[col].nunique()
    sample  = str(df[col].dropna().unique()[:3].tolist())
    print(f'{col:12s}  {dtype:10s}  {n_miss:>8}  {pct:>8.1f}%  {n_uniq:>7}  {sample}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1.4  Missingness heatmap — are missing values structured?
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: missingness per feature, overall
miss_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=True)
colors_bar = ['crimson' if v > 30 else 'steelblue' if v > 5 else '#90caf9'
              for v in miss_pct]
axes[0].barh(miss_pct.index, miss_pct.values, color=colors_bar, edgecolor='white')
axes[0].axvline(5,  color='orange', linestyle='--', lw=1.5, label='>5% — caution')
axes[0].axvline(30, color='crimson', linestyle='--', lw=1.5, label='>30% — high risk')
axes[0].set_title('Missing Data by Feature', fontsize=12)
axes[0].set_xlabel('% Missing')
axes[0].legend()

# Right: missingness by hospital site
miss_by_site = df.groupby('dataset').apply(
    lambda g: g.isnull().mean() * 100
)[['trestbps', 'chol', 'thalch', 'oldpeak', 'slope', 'ca', 'thal']]

im = axes[1].imshow(miss_by_site.values, cmap='Reds', aspect='auto', vmin=0, vmax=100)
axes[1].set_xticks(range(len(miss_by_site.columns)))
axes[1].set_xticklabels(miss_by_site.columns, rotation=30, ha='right', fontsize=9)
axes[1].set_yticks(range(len(miss_by_site.index)))
axes[1].set_yticklabels(miss_by_site.index)
for i in range(len(miss_by_site.index)):
    for j in range(len(miss_by_site.columns)):
        val = miss_by_site.values[i, j]
        axes[1].text(j, i, f'{val:.0f}%', ha='center', va='center',
                     fontsize=9, color='white' if val > 50 else 'black')
plt.colorbar(im, ax=axes[1], label='% Missing')
axes[1].set_title('Missing Data by Hospital Site\n(Site-level MNAR — some tests not routinely ordered)', fontsize=11)

plt.suptitle('Clinical Data Quality Audit — UCI Heart Disease Dataset', fontsize=13)
plt.tight_layout()
plt.show()

print('Key finding: ca and thal are missing in ~95% of Switzerland and VA Long Beach records.')
print('These are structural gaps — those hospitals did not routinely perform fluoroscopy or thalassemia testing.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1.5  Clinical feature distributions — understand what you're modeling
# ─────────────────────────────────────────────────────────────

# Create binary target first for comparison plots
df['target'] = (df['num'] > 0).astype(int)
label_map = {0: 'No Disease', 1: 'Heart Disease'}

fig, axes = plt.subplots(3, 4, figsize=(18, 13))
axes = axes.flatten()

continuous_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak']
binary_cols     = ['fbs', 'exang']
cat_cols        = ['sex', 'cp', 'restecg', 'slope']

palette = {0: 'steelblue', 1: 'crimson'}

idx = 0
# Continuous: overlapping histograms by target
for col in continuous_cols:
    for tgt, color in palette.items():
        subset = df[df['target'] == tgt][col].dropna()
        axes[idx].hist(subset, bins=28, alpha=0.55, color=color,
                       edgecolor='white', label=label_map[tgt])
    axes[idx].set_title(f'{col}', fontsize=11)
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Count')
    if idx == 0:
        axes[idx].legend(fontsize=8)
    idx += 1

# Categorical: grouped bar charts
for col in cat_cols:
    ct = pd.crosstab(df[col], df['target'], normalize='index') * 100
    x  = np.arange(len(ct))
    w  = 0.35
    if 0 in ct.columns:
        axes[idx].bar(x - w/2, ct[0], width=w, color='steelblue', alpha=0.8, label='No Disease')
    if 1 in ct.columns:
        axes[idx].bar(x + w/2, ct[1], width=w, color='crimson',   alpha=0.8, label='Heart Disease')
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(ct.index, rotation=20, ha='right', fontsize=8)
    axes[idx].set_title(f'{col}', fontsize=11)
    axes[idx].set_ylabel('% within category')
    if idx == 5:
        axes[idx].legend(fontsize=8)
    idx += 1

# Binary flags
for col in binary_cols:
    ct = pd.crosstab(df[col], df['target'], normalize='index') * 100
    x  = np.arange(len(ct))
    if 0 in ct.columns:
        axes[idx].bar(x - 0.2, ct[0], width=0.35, color='steelblue', alpha=0.8, label='No Disease')
    if 1 in ct.columns:
        axes[idx].bar(x + 0.2, ct[1], width=0.35, color='crimson',   alpha=0.8, label='Heart Disease')
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(['No', 'Yes'])
    axes[idx].set_title(col, fontsize=11)
    axes[idx].set_ylabel('% within group')
    idx += 1

# Hide unused
for j in range(idx, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Clinical Feature Distributions — No Disease (blue) vs Heart Disease (red)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1.6  Correlation matrix — which features move together?
# ─────────────────────────────────────────────────────────────

# Encode categoricals for correlation analysis
df_corr = df[['age','trestbps','chol','thalch','oldpeak','fbs','exang','target']].copy()
df_corr['cp_asym'] = (df['cp'] == 'asymptomatic').astype(int)
df_corr['sex_male'] = (df['sex'] == 'Male').astype(int)
df_corr = df_corr.apply(pd.to_numeric, errors='coerce')

corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.046)

labels = corr_matrix.columns.tolist()
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
ax.set_yticklabels(labels, fontsize=9)

for i in range(len(labels)):
    for j in range(len(labels)):
        val = corr_matrix.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=7.5, color='white' if abs(val) > 0.5 else 'black')

ax.set_title('Feature Correlation Matrix\nStrong correlations with target guide feature selection',
             fontsize=12)
plt.tight_layout()
plt.show()

print('Strongest correlations with target (heart disease):')
target_corr = corr_matrix['target'].drop('target').sort_values(key=abs, ascending=False)
for feat, val in target_corr.items():
    direction = '↑ risk' if val > 0 else '↓ risk'
    print(f'  {feat:15s}  r = {val:+.3f}   {direction}')

### 1.7 Feature Engineering for Clinical Data

Raw clinical variables often need transformation before they carry maximum predictive power:

**Encoding strategies:**
- **Binary categories** (sex, fbs, exang): map directly to 0/1 — no information lost
- **Nominal categories** (cp, restecg, thal): one-hot encode — avoids implying false ordinal relationships
- **Ordinal categories** (slope: upsloping=0, flat=1, downsloping=2): label encode — the ordering has clinical meaning

**Imputation strategies:**
- **Median imputation** for continuous features with few missing values — robust to outliers vs. mean
- **Mode imputation** for categorical features
- **Missing indicator features** — add a `{col}_missing` binary flag so the model can learn that the absence of a test result is itself informative
- **Drop columns** with > 50% missingness when the missingness is MNAR and non-informative

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1.8  Full feature engineering pipeline
# ─────────────────────────────────────────────────────────────

def engineer_features(df_input):
    """
    Transform raw clinical records into an ML-ready feature matrix.
    Returns: X (DataFrame), y (Series)
    """
    df_e = df_input.copy()

    # ── Target: binarize (0 = healthy, 1 = disease) ──
    y = (df_e['num'] > 0).astype(int)

    # ── Binary flags ──
    df_e['sex_male']  = (df_e['sex'] == 'Male').astype(int)
    df_e['fbs']       = pd.to_numeric(df_e['fbs'],   errors='coerce').fillna(0).astype(int)
    df_e['exang']     = pd.to_numeric(df_e['exang'],  errors='coerce').fillna(0).astype(int)

    # ── Ordinal encoding — slope (clinical risk order) ──
    slope_order = {'upsloping': 0, 'flat': 1, 'downsloping': 2}
    df_e['slope_enc'] = df_e['slope'].map(slope_order)

    # ── One-hot encoding — nominal categories ──
    cp_dummies      = pd.get_dummies(df_e['cp'],      prefix='cp',      drop_first=False)
    restecg_dummies = pd.get_dummies(df_e['restecg'], prefix='restecg', drop_first=False)

    # ── Missing value indicators (informative missingness) ──
    df_e['slope_missing'] = df_e['slope'].isnull().astype(int)
    df_e['ca_missing']    = df_e['ca'].isnull().astype(int)
    df_e['thal_missing']  = df_e['thal'].isnull().astype(int)

    # ── Continuous features (median imputation) ──
    cont_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak']
    for col in cont_cols:
        df_e[col] = df_e[col].fillna(df_e[col].median())

    # ── Impute slope_enc and ca separately ──
    df_e['slope_enc'] = df_e['slope_enc'].fillna(df_e['slope_enc'].median())
    df_e['ca_enc']    = df_e['ca'].fillna(df_e['ca'].median())

    # ── Assemble final feature matrix ──
    base_cols = cont_cols + ['sex_male', 'fbs', 'exang', 'slope_enc', 'ca_enc',
                             'slope_missing', 'ca_missing', 'thal_missing']
    X = pd.concat([df_e[base_cols], cp_dummies, restecg_dummies], axis=1)
    X.columns = X.columns.astype(str)

    return X, y


X, y = engineer_features(df)

print(f'Feature matrix shape: {X.shape}  ({X.shape[0]} patients × {X.shape[1]} features)')
print(f'Target distribution:  {y.value_counts().to_dict()}  ({y.mean()*100:.1f}% heart disease)')
print(f'\nAny NaN remaining: {X.isnull().any().any()}')
print(f'\nFeature list:')
for i, col in enumerate(X.columns, 1):
    print(f'  {i:2d}. {col}')

---
## Part 2 — Binary Classification for Heart Disease

### 2.1 What Is Binary Classification?

**Binary classification** is the task of assigning each input to one of exactly two classes. In our case:

- **Class 0** — No heart disease (patient is healthy)
- **Class 1** — Heart disease present (patient requires intervention)

A classifier learns a **decision boundary** — a surface in feature space that separates the two classes. Every new patient falls on one side of that boundary and is assigned the corresponding label.

---

### 2.2 Why Accuracy Alone Is Not Enough in Medicine

Consider a trivial classifier that always predicts "No Disease" for everyone. On our dataset (55% disease, 45% healthy), this would achieve **45% accuracy** — but it would miss **every single sick patient**. In clinical settings, this is catastrophic.

We need metrics that distinguish between two types of errors:

| Prediction \ Reality | Healthy (0) | Disease (1) |
|---|---|---|
| **Predicted Healthy (0)** | True Negative (TN) ✅ | **False Negative (FN)** ❌ ← missed case |
| **Predicted Disease (1)** | **False Positive (FP)** ⚠️ ← unnecessary workup | True Positive (TP) ✅ |

From these four cells:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

$$\text{Precision} = \frac{TP}{TP + FP} \quad \text{(Of those flagged, how many truly had disease?)}$$

$$\text{Recall (Sensitivity)} = \frac{TP}{TP + FN} \quad \text{(Of all disease cases, how many did we catch?)}$$

$$\text{F1} = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} \quad \text{(Harmonic mean — balances both)}$$

$$\text{Specificity} = \frac{TN}{TN + FP} \quad \text{(Of all healthy patients, how many did we correctly spare?)}$$

**Clinical trade-off:** In disease screening, we typically prioritize **recall (sensitivity)** — it is better to flag a healthy person for follow-up (FP) than to send a sick patient home (FN). The cost of a missed diagnosis almost always outweighs the cost of an unnecessary test.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.3  Train / test split with stratification
# Stratification ensures both splits have the same disease rate
# ─────────────────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Stratified Train/Test Split')
print(f'  Train: {len(y_train)} patients  |  Disease rate: {y_train.mean()*100:.1f}%')
print(f'  Test : {len(y_test)}  patients  |  Disease rate: {y_test.mean()*100:.1f}%')
print(f'  ✓ Disease rates are balanced across splits')

# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

### 2.4 Model 1 — Logistic Regression

**Logistic Regression** is the workhorse of clinical binary classification. Despite the name, it is a **classifier**, not a regressor.

The model computes a weighted sum of features (like linear regression), then squashes the result through the **sigmoid function** to produce a probability between 0 and 1:

$$P(y=1 | \mathbf{x}) = \sigma(\boldsymbol{\theta}^T \mathbf{x}) = \frac{1}{1 + e^{-\boldsymbol{\theta}^T \mathbf{x}}}$$

If $P > 0.5$, the patient is classified as having heart disease. The threshold 0.5 can be adjusted — **lowering it increases recall at the cost of precision**, which is often the right trade-off in clinical screening.

Logistic regression is favored in medicine because:
- The coefficients are interpretable as **log-odds ratios**
- It outputs calibrated **probabilities**, not just labels
- It trains quickly and generalizes well when data is limited

In [ ]:
# ─────────────────────────────────────────────────────────────
# Logistic Regression — train and inspect coefficients
# ─────────────────────────────────────────────────────────────

log_reg = LogisticRegression(max_iter=2000, random_state=42, C=1.0)
log_reg.fit(X_train_s, y_train)

y_pred_lr  = log_reg.predict(X_test_s)
y_prob_lr  = log_reg.predict_proba(X_test_s)[:, 1]

# ── Coefficient interpretation ──
coef_df = pd.DataFrame({
    'feature'    : X.columns,
    'coefficient': log_reg.coef_[0]
}).sort_values('coefficient', key=abs, ascending=True)

# Convert log-odds to odds ratio for clinical interpretability
coef_df['odds_ratio'] = np.exp(coef_df['coefficient'])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: coefficient bar chart
colors_coef = ['crimson' if c > 0 else 'steelblue' for c in coef_df['coefficient']]
axes[0].barh(coef_df['feature'], coef_df['coefficient'],
             color=colors_coef, alpha=0.8, edgecolor='white')
axes[0].axvline(0, color='black', lw=0.8)
axes[0].set_title('Logistic Regression Coefficients\n'
                  'Red = increases disease risk | Blue = decreases risk', fontsize=11)
axes[0].set_xlabel('Log-Odds Coefficient')

# Right: sigmoid curve showing probability output concept
z = np.linspace(-6, 6, 300)
sigma = 1 / (1 + np.exp(-z))
axes[1].plot(z, sigma, color='steelblue', lw=3)
axes[1].axhline(0.5, color='crimson', linestyle='--', lw=2, label='Default threshold = 0.5')
axes[1].fill_between(z[z >= 0], sigma[z >= 0], 0.5, alpha=0.12, color='crimson',
                     label='Predicted: Disease')
axes[1].fill_between(z[z <= 0], sigma[z <= 0], 0.5, alpha=0.12, color='steelblue',
                     label='Predicted: Healthy')
axes[1].set_title('The Sigmoid Function\n'
                  'Squashes any real number into a probability [0, 1]', fontsize=11)
axes[1].set_xlabel('Linear combination (θᵀx)')
axes[1].set_ylabel('P(Heart Disease)')
axes[1].legend()

plt.suptitle('Logistic Regression — How It Works', fontsize=13)
plt.tight_layout()
plt.show()

# Top predictors
print('Top 5 features INCREASING heart disease risk (odds ratio > 1):')
top_pos = coef_df.sort_values('coefficient', ascending=False).head(5)
for _, row in top_pos.iterrows():
    print(f'  {row["feature"]:25s}  coef={row["coefficient"]:+.3f}  OR={row["odds_ratio"]:.2f}x')

print('\nTop 5 features DECREASING heart disease risk (odds ratio < 1):')
top_neg = coef_df.sort_values('coefficient').head(5)
for _, row in top_neg.iterrows():
    print(f'  {row["feature"]:25s}  coef={row["coefficient"]:+.3f}  OR={row["odds_ratio"]:.2f}x')

### 2.5 Model 2 — Decision Tree

A **Decision Tree** partitions the feature space by asking a sequence of binary questions — exactly like a clinical decision flowchart. Each **internal node** tests a feature threshold; each **leaf node** outputs a class prediction.

Trees are uniquely valuable in clinical settings because:
- They mirror how physicians reason ("if chest pain is asymptomatic AND max heart rate < 140, then...")
- They are **fully transparent** — you can print the entire decision logic
- They handle mixed data types natively

The trade-off: unconstrained trees **overfit severely**. A tree that memorizes training data will have 100% training accuracy and much lower test accuracy. Constraining `max_depth` is the primary regularization lever.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Decision Tree — fit a shallow tree and print the rules
# ─────────────────────────────────────────────────────────────

dt = DecisionTreeClassifier(max_depth=4, random_state=42, min_samples_leaf=15)
dt.fit(X_train_s, y_train)

y_pred_dt = dt.predict(X_test_s)
y_prob_dt = dt.predict_proba(X_test_s)[:, 1]

print('Decision Tree Rules (depth 4) — readable clinical logic:')
print('─' * 70)
tree_rules = export_text(dt, feature_names=list(X.columns), max_depth=3)
# Show first 60 lines to keep output readable
lines = tree_rules.split('\n')
for line in lines[:55]:
    print(line)
if len(lines) > 55:
    print(f'  ... ({len(lines)-55} more lines)')

### 2.6 Model 3 — Random Forest

A **Random Forest** is an ensemble of many decision trees, each trained on a random bootstrap sample of the data and a random subset of features. The final prediction is the **majority vote** across all trees.

This technique — called **bagging** (Bootstrap AGGregatING) — dramatically reduces variance. Each individual tree may overfit its bootstrap sample, but averaging across hundreds of trees cancels out the idiosyncratic errors. Random forests are consistently among the strongest out-of-box classifiers for tabular clinical data.

The key hyperparameters:
- `n_estimators` — number of trees (more = better, with diminishing returns)
- `max_depth` — maximum depth of each tree
- `max_features` — number of features considered at each split (controls tree diversity)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Random Forest — train and extract feature importances
# ─────────────────────────────────────────────────────────────

rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42,
                            min_samples_leaf=5, n_jobs=-1)
rf.fit(X_train_s, y_train)

y_pred_rf = rf.predict(X_test_s)
y_prob_rf = rf.predict_proba(X_test_s)[:, 1]

# Feature importance plot
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 8))
colors_imp = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(importances)))
bars = ax.barh(importances.index, importances.values,
               color=colors_imp, edgecolor='white')
ax.set_title('Random Forest Feature Importances\n'
             '(Mean Decrease in Impurity across 300 trees)', fontsize=12)
ax.set_xlabel('Importance Score')

# Annotate top 3
for i, (name, val) in enumerate(importances.tail(3).items()):
    ax.text(val + 0.002, len(importances) - 3 + i,
            f'  {val:.3f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print('Top 5 most important clinical features (Random Forest):')
for feat, imp in importances.tail(5).iloc[::-1].items():
    print(f'  {feat:30s}  importance = {imp:.4f}')

### 2.7 Evaluating Clinical Classifiers

We now evaluate all three models using the full battery of clinical metrics.

**The confusion matrix** is the foundation — it makes every type of error explicit. From it, we can compute:

- **Sensitivity (Recall):** Did we catch the sick patients? Critical for screening.
- **Specificity:** Did we correctly clear the healthy patients? Reduces unnecessary procedures.
- **PPV (Precision):** When we say "disease", how often are we right? Determines downstream workup burden.
- **AUC-ROC:** Threshold-independent measure of overall discriminative ability. AUC = 1.0 is perfect; 0.5 is random.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Confusion matrices for all three models side by side
# ─────────────────────────────────────────────────────────────

models_eval = [
    ('Logistic Regression', y_pred_lr, y_prob_lr),
    ('Decision Tree',       y_pred_dt, y_prob_dt),
    ('Random Forest',       y_pred_rf, y_prob_rf),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, y_pred, _) in zip(axes, models_eval):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['No Disease', 'Heart Disease'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')

    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    ax.set_title(f'{name}\nSensitivity={sens:.2f}  Specificity={spec:.2f}', fontsize=10)

plt.suptitle('Confusion Matrices — What Kind of Errors Does Each Model Make?', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# Full metrics comparison table + ROC curves
# ─────────────────────────────────────────────────────────────

def clinical_metrics(name, y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'Model'      : name,
        'Accuracy'   : accuracy_score(y_true, y_pred),
        'Sensitivity': tp / (tp + fn),   # recall
        'Specificity': tn / (tn + fp),
        'Precision'  : precision_score(y_true, y_pred),
        'F1'         : f1_score(y_true, y_pred),
        'AUC-ROC'    : roc_auc_score(y_true, y_prob),
        'FN (missed)': int(fn),
        'FP (alarm)' : int(fp),
    }

results = [clinical_metrics(n, y_test, yp, ypr) for n, yp, ypr in models_eval]
results_df = pd.DataFrame(results).set_index('Model')

print('=== Clinical Classifier Performance Summary ===')
print(results_df.round(4).to_string())

# ── ROC Curves ──
fig, ax = plt.subplots(figsize=(8, 7))

roc_colors = ['steelblue', 'darkorange', 'green']
for (name, _, y_prob), color in zip(models_eval, roc_colors):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f'{name}  (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random classifier (AUC = 0.50)')
ax.fill_between([0, 1], [0, 1], 0, alpha=0.04, color='gray')

# Annotate the clinical "sweet spot" — high sensitivity quadrant
ax.axhline(0.80, color='crimson', linestyle=':', lw=1, alpha=0.5)
ax.text(0.6, 0.82, 'Sensitivity ≥ 0.80 target', fontsize=9, color='crimson')

ax.set_title('ROC Curves — Heart Disease Classification\n'
             'Higher and to the left = better discrimination', fontsize=12)
ax.set_xlabel('False Positive Rate (1 − Specificity)')
ax.set_ylabel('True Positive Rate (Sensitivity)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 2.8 The Decision Threshold — Trading Sensitivity for Specificity

All three models output a **probability** between 0 and 1. The default decision threshold is 0.5: patients above it are classified as disease-positive.

But 0.5 is not always clinically appropriate. In **population screening**, we want to catch as many true cases as possible — we are willing to accept more false positives (patients who get a follow-up test they didn't need) in exchange for fewer false negatives (patients who are incorrectly told they are healthy).

**Lowering the threshold** (e.g., to 0.3) increases sensitivity but reduces specificity. **Raising it** does the opposite. The right threshold depends on the relative cost of FN vs FP in the clinical context — which is a medical and economic decision, not just a statistical one.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Threshold analysis — sensitivity/specificity vs decision cutoff
# Using the Random Forest (best AUC)
# ─────────────────────────────────────────────────────────────

thresholds = np.linspace(0.01, 0.99, 200)
sens_list, spec_list, prec_list, f1_list = [], [], [], []

for t in thresholds:
    y_pred_t = (y_prob_rf >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)
    tn, fp, fn, tp = cm.ravel()
    sens_list.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    spec_list.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    prec_list.append(tp / (tp + fp) if (tp + fp) > 0 else 0)
    f1_list.append(f1_score(y_test, y_pred_t, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: all metrics vs threshold
axes[0].plot(thresholds, sens_list, color='crimson',    lw=2.5, label='Sensitivity (Recall)')
axes[0].plot(thresholds, spec_list, color='steelblue',  lw=2.5, label='Specificity')
axes[0].plot(thresholds, prec_list, color='darkorange',  lw=2.5, label='Precision (PPV)')
axes[0].plot(thresholds, f1_list,   color='green',       lw=2.5, label='F1 Score')

# Mark optimal F1 threshold
best_thresh = thresholds[np.argmax(f1_list)]
axes[0].axvline(0.5, color='gray', linestyle=':', lw=1.5, label='Default (0.5)')
axes[0].axvline(best_thresh, color='purple', linestyle='--', lw=2,
                label=f'Best F1 threshold = {best_thresh:.2f}')
axes[0].set_title('Metric vs Decision Threshold\n(Random Forest)', fontsize=11)
axes[0].set_xlabel('Decision Threshold')
axes[0].set_ylabel('Score')
axes[0].legend(fontsize=8)

# Right: compare confusion matrices at threshold 0.5 vs 0.3
for ax_sub, thresh, title_suffix in [(axes[1], 0.35, 'Low threshold = 0.35\n(Higher sensitivity, more false alarms)')]:
    y_pred_low = (y_prob_rf >= thresh).astype(int)
    cm_low = confusion_matrix(y_test, y_pred_low)
    tn, fp, fn, tp = cm_low.ravel()
    disp = ConfusionMatrixDisplay(cm_low, display_labels=['No Disease', 'Heart Disease'])
    disp.plot(ax=ax_sub, colorbar=False, cmap='Reds')
    sens_low = tp / (tp + fn)
    spec_low = tn / (tn + fp)
    ax_sub.set_title(f'Random Forest at threshold={thresh}\nSensitivity={sens_low:.2f} Specificity={spec_low:.2f}\n{title_suffix}',
                     fontsize=9)

plt.suptitle('Threshold Tuning — The Clinical Trade-Off Between Sensitivity and Specificity', fontsize=12)
plt.tight_layout()
plt.show()

print(f'At threshold = 0.50:  Sensitivity = {sens_list[100]:.3f}  |  Specificity = {spec_list[100]:.3f}')
print(f'At threshold = 0.35:  Sensitivity = {sens_list[np.searchsorted(thresholds, 0.35)]:.3f}  |  Specificity = {spec_list[np.searchsorted(thresholds, 0.35)]:.3f}')
print(f'\nFor screening (catching all sick patients), prefer threshold = 0.35')
print(f'For confirmation (high confidence before invasive procedure), prefer threshold = 0.65+')

### 2.9 Cross-Validated Performance — Honest Estimates

A single train/test split can give an optimistic or pessimistic estimate depending on which patients ended up in the test set. For a clinical tool that will be deployed on future patients, we need **stable, unbiased performance estimates** using stratified cross-validation.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5-fold stratified cross-validation across all models
# ─────────────────────────────────────────────────────────────

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Logistic Regression': Pipeline([('sc', StandardScaler()),
                                      ('lr', LogisticRegression(max_iter=2000, C=1.0, random_state=42))]),
    'Decision Tree (d=4)': Pipeline([('sc', StandardScaler()),
                                      ('dt', DecisionTreeClassifier(max_depth=4, min_samples_leaf=15, random_state=42))]),
    'Random Forest':       Pipeline([('sc', StandardScaler()),
                                      ('rf', RandomForestClassifier(n_estimators=300, max_depth=8,
                                                                     min_samples_leaf=5, random_state=42, n_jobs=-1))]),
}

print('5-Fold Stratified Cross-Validation Results')
print('=' * 72)
print(f'{"Model":25s}  {"AUC-ROC":>12}  {"Sensitivity":>13}  {"Specificity":>13}  {"F1":>8}')
print('-' * 72)

cv_summary = {}
for name, pipe in cv_models.items():
    auc   = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
    sens  = cross_val_score(pipe, X, y, cv=cv, scoring='recall')
    prec  = cross_val_score(pipe, X, y, cv=cv, scoring='precision')
    f1    = cross_val_score(pipe, X, y, cv=cv, scoring='f1')
    spec_scores = []
    for tr, val in cv.split(X, y):
        pipe.fit(X.iloc[tr], y.iloc[tr])
        yp = pipe.predict(X.iloc[val])
        cm = confusion_matrix(y.iloc[val], yp)
        tn, fp, fn, tp = cm.ravel()
        spec_scores.append(tn / (tn + fp))
    spec = np.array(spec_scores)

    cv_summary[name] = {'AUC': auc, 'Sens': sens, 'Spec': spec, 'F1': f1}
    print(f'{name:25s}  '
          f'{auc.mean():.3f} ± {auc.std():.3f}  '
          f'{sens.mean():.3f} ± {sens.std():.3f}  '
          f'{spec.mean():.3f} ± {spec.std():.3f}  '
          f'{f1.mean():.3f} ± {f1.std():.3f}')

print()
best_model = max(cv_summary, key=lambda k: cv_summary[k]['AUC'].mean())
print(f'Best model by AUC-ROC: {best_model}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cross-validation stability — violin plots per metric
# ─────────────────────────────────────────────────────────────

metrics_to_plot = ['AUC', 'Sens', 'Spec', 'F1']
metric_labels   = ['AUC-ROC', 'Sensitivity', 'Specificity', 'F1']

fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=False)

for ax, metric, label in zip(axes, metrics_to_plot, metric_labels):
    data_vp  = [cv_summary[n][metric] for n in cv_models]
    model_ns = list(cv_models.keys())
    vp = ax.violinplot(data_vp, positions=range(len(model_ns)), showmeans=True)

    colors_vp = ['steelblue', 'darkorange', 'green']
    for body, color in zip(vp['bodies'], colors_vp):
        body.set_facecolor(color)
        body.set_alpha(0.65)

    ax.set_xticks(range(len(model_ns)))
    ax.set_xticklabels([n.replace(' ', '\n') for n in model_ns], fontsize=8)
    ax.set_title(label, fontsize=11)
    ax.set_ylabel('Score')

    # Annotate means
    for i, d in enumerate(data_vp):
        ax.text(i, d.mean() + 0.015, f'{d.mean():.3f}', ha='center',
                fontsize=8.5, fontweight='bold')

plt.suptitle('5-Fold CV Score Distributions — Stability of Clinical Classifiers\n'
             'Taller violin = more variance across folds = less stable model',
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# Final comprehensive summary
# ─────────────────────────────────────────────────────────────

# Retrain best model on full training set, report final test performance
best_pipe_cv = cv_models[best_model]
best_pipe_cv.fit(X_train, y_train)
y_final_pred = best_pipe_cv.predict(X_test)
y_final_prob = best_pipe_cv.predict_proba(X_test)[:, 1]

print(f'FINAL MODEL: {best_model}')
print('=' * 55)
print(classification_report(y_test, y_final_pred,
                             target_names=['No Disease', 'Heart Disease']))

cm_final = confusion_matrix(y_test, y_final_pred)
tn, fp, fn, tp = cm_final.ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
ConfusionMatrixDisplay(cm_final, display_labels=['No Disease', 'Heart Disease']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Final Model: {best_model}\n'
                  f'Sensitivity={tp/(tp+fn):.3f}  Specificity={tn/(tn+fp):.3f}\n'
                  f'FN={fn} missed cases  |  FP={fp} false alarms',
                  fontsize=10)

# Probability distribution by true class
axes[1].hist(y_final_prob[y_test == 0], bins=30, alpha=0.6, color='steelblue', label='True: No Disease')
axes[1].hist(y_final_prob[y_test == 1], bins=30, alpha=0.6, color='crimson',   label='True: Heart Disease')
axes[1].axvline(0.5, color='black', linestyle='--', lw=2, label='Threshold = 0.5')
axes[1].set_title('Predicted Probability Distribution by True Class\n'
                  'Good separation → model discriminates well', fontsize=10)
axes[1].set_xlabel('P(Heart Disease)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Final Model Evaluation — UCI Heart Disease', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nAUC-ROC : {roc_auc_score(y_test, y_final_prob):.4f}')
print(f'Accuracy: {accuracy_score(y_test, y_final_pred):.4f}')
print(f'\nClinical interpretation:')
print(f'  Of {tp+fn} true heart disease patients, we correctly identified {tp} ({tp/(tp+fn)*100:.0f}%)')
print(f'  Of {tn+fp} healthy patients, we correctly cleared {tn} ({tn/(tn+fp)*100:.0f}%)')
print(f'  {fn} patients with disease were missed (sent home without diagnosis)')
print(f'  {fp} healthy patients were unnecessarily flagged (extra workup)')

---
## Summary

### Clinical Data Structures

Real clinical datasets are messy by design. They contain four types of features (continuous, binary, nominal, ordinal) each requiring different treatment. Missing data is rarely random — in the UCI Heart Disease dataset, 66% of `ca` and 53% of `thal` values are absent because certain hospitals never ordered those tests. Recognizing this structural pattern prevented us from naively imputing or dropping rows. We built missing-value indicator features so the model could learn from the absence of a test as a signal in its own right.

### Binary Classification

We framed the problem as binary classification (disease present vs. absent) and applied three models of increasing complexity: Logistic Regression, Decision Tree, and Random Forest.

The central lesson is that **accuracy is the wrong metric for clinical classifiers**. A missed heart disease patient (False Negative) carries a fundamentally different cost than a false alarm (False Positive). Sensitivity, specificity, and AUC-ROC give a complete picture. The decision threshold is not fixed at 0.5 — it is a clinical judgment about the relative cost of each error type, and should be set based on the deployment context.

Cross-validation on stratified folds gave stable, honest performance estimates that a single train/test split cannot provide.

---

### The Clinical ML Checklist

```
1. Audit every feature — type, missingness, clinical meaning
2. Understand WHY data is missing — MCAR / MAR / MNAR
3. Encode categoricals correctly — nominal → one-hot, ordinal → label
4. Add missingness indicators for structurally absent features
5. Stratify the train/test split to preserve disease prevalence
6. Evaluate with AUC, sensitivity, specificity — not just accuracy
7. Tune the decision threshold for the clinical use case
8. Use stratified cross-validation for honest performance estimates
9. Inspect model logic — coefficients, tree rules, feature importance
```

---
*Dataset: UCI Heart Disease Database — Cleveland, Hungary, Switzerland, VA Long Beach (920 patients)*  
*Janosi, A., Steinbrunn, W., Pfisterer, M., Detrano, R. (1988)*